# Binomial Tree and Risk-Neutral Pricing

An option's value cannot be found by discounting its expected payoff: whose expectation, and at which rate? The binomial model answers
both questions with a single argument. If the option can be replicated by trading the stock and a bond, its price must equal the cost of
the replicating portfolio, otherwise there is an arbitrage. That cost turns out to be an expectation under a special set of probabilities,
the risk-neutral ones, discounted at the risk-free rate. The real probability of the stock going up does not appear at all. This
notebook builds the argument in one period, extends it to many periods with the binomial distribution, and prices European and
American options.

## 0. Math link

- [Binomial distribution](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/01_probability_distribution/03_binomial_distribution.ipynb)
  and [binomial coefficient](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/01_probability_distribution/04_binomial_coefficient.ipynb):
  the number of up-moves after $n$ steps is binomial, which gives a closed form for European option prices.
- [Systems of linear equations](https://github.com/dnlwrthstr/LearningLinearAlgebra/tree/main/hyperskill/03_systems_of_linear_equations):
  replication solves two equations in two unknowns (shares and bonds).
- Continuous compounding from module 00 (planned): $R = e^{r\Delta t}$.

## 1. Introduction

A trader sells a one-year call on a Swiss stock and wants to know the price and how to hedge it. The binomial model gives both: the hedge
is the replicating portfolio, and the price is its cost. Because the hedge removes all risk, nobody's view of the stock's expected
return matters.

After this notebook you can:

- replicate an option in a one-period model and derive the risk-neutral probability $q$;
- explain why the real probability of an up-move does not affect the price, and when arbitrage exists;
- price European options in an $n$-step tree by backward induction and by the binomial formula;
- calibrate the tree to a volatility (Cox–Ross–Rubinstein) and watch it converge;
- price American options with early exercise and find the exercise boundary;
- check put–call parity.

## 2. Theory

### One period

The stock is worth $S$ today and $uS$ or $dS$ after one period; a bond grows by the factor $R$ (e.g. $R = e^{r\Delta t}$). An option pays $C_u$ or $C_d$.

**Replication.** Buy $\Delta$ shares and invest $B$ in the bond so that the portfolio pays the same as the option in both states:

$$
\Delta uS + BR = C_u, \qquad \Delta dS + BR = C_d
$$

Two equations, two unknowns:

$$
\Delta = \frac{C_u - C_d}{(u - d)S}, \qquad B = \frac{uC_d - dC_u}{(u - d)R}
$$

**No arbitrage.** If the option traded at a price other than $\Delta S + B$, buying the cheaper of the two (option or portfolio) and selling the
other would lock in a riskless profit. Hence

$$
C = \Delta S + B = \frac{1}{R}\big[qC_u + (1-q)C_d\big], \qquad q = \frac{R - d}{u - d}
$$

**Risk-neutral probability.** If $d < R < u$, then $0 < q < 1$, and $C$ is the discounted expectation of the payoff under the probabilities $(q, 1-q)$. Under $q$
the stock itself earns the risk-free rate: $qu + (1-q)d = R$. If $R \le d$ or $R \ge u$, the stock or the bond dominates the other and there is an
arbitrage.

The real probability $p$ of an up-move appears nowhere. A higher $p$ raises the stock's expected return, but the stock price $S$ already reflects it,
and the option's value relative to the stock does not change.

### Many periods

With $n$ periods of length $\Delta t = T/n$, the stock after $k$ up-moves and $n-k$ down-moves is $Su^kd^{n-k}$. Working backward from the payoff, each node is the
discounted risk-neutral expectation of its two successors. For a European option this collapses to

$$
C = R^{-n}\sum_{k=0}^n\binom nk q^k(1-q)^{n-k}\,\text{payoff}\big(Su^kd^{n-k}\big)
$$

an expectation over a binomial distribution of the number of up-moves.

### Cox–Ross–Rubinstein calibration

To match an annual volatility $\sigma$: $u = e^{\sigma\sqrt{\Delta t}}$, $d = 1/u$, $R = e^{r\Delta t}$. Then the log return per step has variance close to $\sigma^2\Delta t$ under $q$, and as $n \to \infty$ the
distribution of $\ln S_T$ becomes normal (central limit theorem, 10/02) and the tree price converges to the Black–Scholes price (10/05).

### American options

An American option can be exercised at any node. At each node the value is

$$
V = \max\Big(\text{exercise value},\ \frac{1}{R}\big[qV_u + (1-q)V_d\big]\Big)
$$

An American call on a stock without dividends is never exercised early (exercising throws away time value and the interest on $K$), so it equals
the European call. An American put can be worth exercising when the stock is low: receiving $K$ now earns interest.

### Put–call parity

For European options with the same strike and maturity, $C - P = S - KR^{-n}$: a call minus a put is a forward. It holds in every node and at
every $n$.

## 3. Worked Examples

### Example 1: one period

$S = 100$ (CHF), $u = 1.2$, $d = 0.9$, $R = 1.05$, call with strike 100: $C_u = 20$, $C_d = 0$.

$q = (1.05 - 0.9)/(1.2 - 0.9) = 0.5$, $C = 0.5\cdot 20/1.05 = 9.524$.

Replication: $\Delta = 20/30 = 0.667$ shares, $B = (1.2\cdot 0 - 0.9\cdot 20)/(0.3\cdot 1.05) = -57.14$ (borrowing). Cost $66.67 - 57.14 = 9.52$. Check: up state $0.667\cdot 120 - 57.14\cdot 1.05 = 80 - 60 = 20$; down state
$0.667\cdot 90 - 60 = 0$.

If the real probability of the up-move is 80%, the expected payoff is 16 and its value discounted at the risk-free rate would be 15.24, but the
price is still 9.52.

### Example 2: arbitrage

If the call trades at 11: sell it, buy the replicating portfolio for 9.52, keep 1.48. At expiry the portfolio pays exactly what the call costs.

### Example 3: two periods

Same $u, d, R$, two steps. Final stock prices 144, 108, 81; call payoffs 44, 8, 0. Backward:
$C_u = (0.5\cdot 44 + 0.5\cdot 8)/1.05 = 24.762$, $C_d = (0.5\cdot 8 + 0.5\cdot 0)/1.05 = 3.810$, $C_0 = (0.5\cdot 24.762 + 0.5\cdot 3.810)/1.05 = 13.605$.
Binomial formula: $1.05^{-2}(0.25\cdot 44 + 0.5\cdot 8 + 0.25\cdot 0) = 15/1.1025 = 13.605$.

### Example 4: an American put

Put with strike 100 in the two-period tree: payoffs 0, 0, 19. European: $P_d = 0.5\cdot 19/1.05 = 9.048$, $P_0 = 0.5\cdot 9.048/1.05 = 4.308$.
American: at the down node, exercising pays $100 - 90 = 10 > 9.048$, so $P_d = 10$ and $P_0 = 0.5\cdot 10/1.05 = 4.762$.

Put–call parity for the European options: $13.605 - 4.308 = 9.297 = 100 - 100/1.1025$.

## 4. Finance Application

### Where it matters

- **Pricing and hedging.** Banks price and hedge options by replication; the hedge ratio $\Delta$ in each node is the delta used on trading desks.
- **American and path-dependent options.** Trees handle early exercise (American puts, callable and convertible bonds, employee options)
  where closed forms do not exist.
- **Real options.** The option to expand, delay or abandon a project is valued with the same backward induction.
- **Interest-rate trees.** Short-rate trees (Black–Derman–Toy, Hull–White) price bonds with embedded options.

### The usual misreadings

- "The call is cheap because the stock will probably go up." The real probability does not enter the price; it is already in the stock price.
- "Risk-neutral probabilities are investors' beliefs." They are pricing weights that make discounted prices fair; they differ from real
  probabilities by a risk adjustment.
- "Discount the expected payoff at the stock's expected return." That rate is wrong for the option, whose risk differs from the stock's and changes
  over time; replication avoids the question.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import comb

rng = np.random.default_rng(42)

### One period, replication and the irrelevance of $p$

In [ ]:
S0, u, d, R, K = 100.0, 1.2, 0.9, 1.05, 100.0
Cu, Cd = max(u * S0 - K, 0), max(d * S0 - K, 0)
delta = (Cu - Cd) / ((u - d) * S0)
B = (u * Cd - d * Cu) / ((u - d) * R)
q = (R - d) / (u - d)
print(f"delta {delta:.4f} shares, bond {B:.2f} CHF, replication cost {delta * S0 + B:.4f}")
print(f"risk-neutral q {q:.3f}, discounted q-expectation {(q * Cu + (1 - q) * Cd) / R:.4f}")
for p in [0.3, 0.5, 0.8]:
    mu_stock = p * u + (1 - p) * d - 1
    print(f"real p {p}: stock expected return {mu_stock:6.1%}, option price still {delta * S0 + B:.4f}, "
          f"naive p-expectation discounted at r {(p * Cu + (1 - p) * Cd) / R:.4f}")

The replication cost and the discounted $q$-expectation are both 9.5238 whatever the real probability. Discounting the real-world expected payoff
at the risk-free rate would give anything from 5.71 to 15.24 depending on $p$; only with $p = q$ does it agree.

### An $n$-step tree

In [ ]:
def binomial_price(S, K, T, r, sigma, n, kind="call", american=False):
    dt = T / n
    u_, d_ = np.exp(sigma * np.sqrt(dt)), np.exp(-sigma * np.sqrt(dt))
    R_ = np.exp(r * dt)
    q_ = (R_ - d_) / (u_ - d_)
    k = np.arange(n + 1)
    ST = S * u_**k * d_ ** (n - k)
    payoff = np.maximum(ST - K, 0) if kind == "call" else np.maximum(K - ST, 0)
    V = payoff
    boundary = []
    for step in range(n - 1, -1, -1):
        k = np.arange(step + 1)
        S_nodes = S * u_**k * d_ ** (step - k)
        V = (q_ * V[1:] + (1 - q_) * V[:-1]) / R_
        if american:
            ex = np.maximum(S_nodes - K, 0) if kind == "call" else np.maximum(K - S_nodes, 0)
            early = ex > V
            if kind == "put" and early.any():
                boundary.append((step * dt, S_nodes[early].max()))
            V = np.maximum(V, ex)
    return (V[0], boundary) if american else V[0]


def binomial_formula(S, K, T, r, sigma, n, kind="call"):
    dt = T / n
    u_, d_ = np.exp(sigma * np.sqrt(dt)), np.exp(-sigma * np.sqrt(dt))
    R_ = np.exp(r * dt)
    q_ = (R_ - d_) / (u_ - d_)
    k = np.arange(n + 1)
    ST = S * u_**k * d_ ** (n - k)
    payoff = np.maximum(ST - K, 0) if kind == "call" else np.maximum(K - ST, 0)
    return R_ ** (-n) * np.sum(stats.binom.pmf(k, n, q_) * payoff)


def black_scholes(S, K, T, r, sigma, kind="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == "call":
        return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)


S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.01, 0.20       # CHF, years, continuous rate, annual volatility
for n in [1, 2, 10, 100, 1000]:
    c_tree = binomial_price(S, K, T, r, sigma, n)
    print(f"n = {n:4d}: call {c_tree:.4f} (formula {binomial_formula(S, K, T, r, sigma, n):.4f})")
print(f"Black-Scholes reference (10/05): {black_scholes(S, K, T, r, sigma):.4f}")

Backward induction and the binomial formula agree for every $n$. With 1,000 steps the tree is within 0.002 of the Black–Scholes price of 8.4333.

### The two-period examples in code

In [ ]:
def tree_explicit(S, u_, d_, R_, K, n, kind="call", american=False):
    q_ = (R_ - d_) / (u_ - d_)
    k = np.arange(n + 1)
    V = np.maximum(S * u_**k * d_ ** (n - k) - K, 0) if kind == "call" else np.maximum(K - S * u_**k * d_ ** (n - k), 0)
    for step in range(n - 1, -1, -1):
        k = np.arange(step + 1)
        V = (q_ * V[1:] + (1 - q_) * V[:-1]) / R_
        if american:
            ex = np.maximum(S * u_**k * d_ ** (step - k) - K, 0) if kind == "call" else np.maximum(K - S * u_**k * d_ ** (step - k), 0)
            V = np.maximum(V, ex)
    return V[0]

c2 = tree_explicit(100, 1.2, 0.9, 1.05, 100, 2)
p2 = tree_explicit(100, 1.2, 0.9, 1.05, 100, 2, "put")
p2a = tree_explicit(100, 1.2, 0.9, 1.05, 100, 2, "put", american=True)
print(f"call {c2:.3f}, European put {p2:.3f}, American put {p2a:.3f}")
print(f"put-call parity: C - P = {c2 - p2:.3f}, S - K/R^2 = {100 - 100 / 1.05**2:.3f}")

### American options and the exercise boundary

In [ ]:
n_am = 500
put_eu = binomial_price(S, K, T, r, sigma, n_am, "put")
put_am, boundary = binomial_price(S, K, T, r, sigma, n_am, "put", american=True)
call_am, _ = binomial_price(S, K, T, r, sigma, n_am, "call", american=True)
call_eu = binomial_price(S, K, T, r, sigma, n_am, "call")
print(f"put:  European {put_eu:.4f}, American {put_am:.4f}, early-exercise premium {put_am - put_eu:.4f}")
print(f"call: European {call_eu:.4f}, American {call_am:.4f}")

r_hi = 0.05
put_eu_hi = binomial_price(S, K, T, r_hi, sigma, n_am, "put")
put_am_hi, boundary_hi = binomial_price(S, K, T, r_hi, sigma, n_am, "put", american=True)
print(f"with r = 5%: European put {put_eu_hi:.4f}, American {put_am_hi:.4f}, premium {put_am_hi - put_eu_hi:.4f}")

The American call equals the European call, as the theory says. The American put is worth 0.076 more than the European one at a 1% interest rate and
0.52 more at 5%: early exercise pays off by earning interest on the strike sooner, so its value rises with the rate.

### Dynamic replication along a path

Follow one simulated path through a 50-step tree (real up-probability 0.6), rebalancing the delta hedge at every node. The hedge portfolio is
self-financing: no money is added or withdrawn after the start.

In [ ]:
def replicate_path(S, K, T, r, sigma, n, p_real, rng_):
    dt = T / n
    u_, d_ = np.exp(sigma * np.sqrt(dt)), np.exp(-sigma * np.sqrt(dt))
    R_ = np.exp(r * dt)
    q_ = (R_ - d_) / (u_ - d_)
    # full tree of option values: V[step][k]
    values = [None] * (n + 1)
    k = np.arange(n + 1)
    values[n] = np.maximum(S * u_**k * d_ ** (n - k) - K, 0)
    for step in range(n - 1, -1, -1):
        values[step] = (q_ * values[step + 1][1:] + (1 - q_) * values[step + 1][:-1]) / R_
    k_up = 0
    port = values[0][0]                                   # start with the option price
    for step in range(n):
        S_now = S * u_**k_up * d_ ** (step - k_up)
        Vu, Vd = values[step + 1][k_up + 1], values[step + 1][k_up]
        delta_ = (Vu - Vd) / (S_now * (u_ - d_))
        bond = port - delta_ * S_now
        up = rng_.random() < p_real
        k_up += up
        S_next = S * u_**k_up * d_ ** (step + 1 - k_up)
        port = delta_ * S_next + bond * R_
    ST = S * u_**k_up * d_ ** (n - k_up)
    return port, max(ST - K, 0), ST

for i in range(3):
    port, payoff, ST = replicate_path(S, K, T, r, sigma, 50, 0.6, rng)
    print(f"path {i + 1}: S_T {ST:7.2f}, option payoff {payoff:6.3f}, hedge portfolio {port:6.3f}")

On every path the self-financing hedge ends with exactly the option payoff, whether the stock finishes slightly above the strike or far above it.
That is the sense in which the option price is the cost of its hedge.

## 6. Visualizations

In [ ]:
def plot_tree(ax, S, u_, d_, R_, K, n):
    q_ = (R_ - d_) / (u_ - d_)
    vals = [None] * (n + 1)
    k = np.arange(n + 1)
    vals[n] = np.maximum(S * u_**k * d_ ** (n - k) - K, 0)
    for step in range(n - 1, -1, -1):
        vals[step] = (q_ * vals[step + 1][1:] + (1 - q_) * vals[step + 1][:-1]) / R_
    for step in range(n + 1):
        for kk in range(step + 1):
            y = 2 * kk - step
            s_node = S * u_**kk * d_ ** (step - kk)
            if step < n:
                ax.plot([step, step + 1], [y, y + 1], color="grey", linewidth=0.6)
                ax.plot([step, step + 1], [y, y - 1], color="grey", linewidth=0.6)
            ax.text(step, y, f"{s_node:.1f}\n{vals[step][kk]:.2f}", ha="center", va="center", fontsize=8,
                    bbox=dict(boxstyle="round", fc="white", ec="tab:blue"))
    ax.set_xticks(range(n + 1))
    ax.set_yticks([])
    ax.set_xlabel("step")

fig, ax = plt.subplots(figsize=(10, 6))
plot_tree(ax, 100, 1.2, 0.9, 1.05, 100, 4)
ax.set_title("Four-step tree: stock price (top) and call value (bottom), u = 1.2, d = 0.9, R = 1.05, K = 100")
plt.show()

In [ ]:
ns = np.arange(1, 201)
prices = [binomial_price(S, K, T, r, sigma, n_) for n_ in ns]
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ns, prices, linewidth=0.8, label="CRR binomial price")
ax.axhline(black_scholes(S, K, T, r, sigma), color="black", linestyle="--", label="Black-Scholes")
ax.set_ylim(7.5, 9.5)
ax.set_title("At-the-money call (S = K = 100, sigma 20%, r 1%, T = 1): convergence of the tree")
ax.set_xlabel("number of steps n")
ax.set_ylabel("price (CHF)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for b, lab, c in [(boundary, "r = 1%", "tab:blue"), (boundary_hi, "r = 5%", "tab:red")]:
    if b:
        tb, sb = np.array(b).T
        order = np.argsort(tb)
        ax.plot(tb[order][::2], sb[order][::2], color=c, label=f"exercise boundary, {lab}")   # every second step: same node parity
ax.axhline(K, color="black", linewidth=0.8, label="strike")
ax.set_title("American put: exercise early when the stock is below the boundary")
ax.set_xlabel("time (years)")
ax.set_ylabel("stock price (CHF)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
n_d = 50
dt = T / n_d
u_, d_ = np.exp(sigma * np.sqrt(dt)), np.exp(-sigma * np.sqrt(dt))
q_ = (np.exp(r * dt) - d_) / (u_ - d_)
p_real = (np.exp(0.07 * dt) - d_) / (u_ - d_)                # real-world probability for a 7% expected return
k = np.arange(n_d + 1)
ST = S * u_**k * d_ ** (n_d - k)
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(ST, stats.binom.pmf(k, n_d, q_), width=2.0, alpha=0.6, label=f"risk-neutral (q = {q_:.3f})")
ax.bar(ST, stats.binom.pmf(k, n_d, p_real), width=2.0, alpha=0.6, label=f"real world, 7% expected return (p = {p_real:.3f})")
ax.axvline(K, color="black", linewidth=0.8)
ax.set_xlim(40, 220)
ax.set_title("Distribution of the stock price after one year, 50-step tree")
ax.set_xlabel("S_T (CHF)")
ax.set_ylabel("probability")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 7. Exercises

### Basic

1. In the one-period model of Example 1, price a put with strike 100 and find its replicating portfolio. Check put–call parity.
2. Suppose $u = 1.2$, $d = 0.9$ and $R = 1.25$. Show how to make a riskless profit with the stock and the bond.
3. A colleague argues: "This call is too expensive. Our analysts expect the stock to rise with 80% probability, so the option will almost surely pay
   off." **Interpret** this argument in the light of the replication result.

### Intermediate

4. Show numerically that the American call on a non-dividend stock equals the European call for several strikes and rates. Then add a dividend: let
   the stock drop by 3% at the step closest to half a year, and compare American and European calls again.
5. Check put–call parity in the CRR tree for $n = 10, 100, 1000$ and strikes 80, 100, 120.

### Advanced

6. Under $q$, compute the mean and variance of the one-step log return $\ln(S_{t+1}/S_t)$ in the CRR tree. Show that the variance is $\sigma^2\Delta t$ up to terms of order $\Delta t^2$,
   and that the mean is $(r - \sigma^2/2)\Delta t$ up to the same order.
7. Price a digital call (pays CHF 1 if $S_T > K$) with the tree for $n = 1..200$ and compare with $e^{-rT}N(d_2)$. Why does the tree price oscillate more than for the ordinary call?

## 8. Solutions / Checks

In [ ]:
# 1.
Pu, Pd = max(K - u * S0, 0), max(K - d * S0, 0)
dP = (Pu - Pd) / ((u - d) * S0)
BP = (u * Pd - d * Pu) / ((u - d) * R)
P1 = dP * S0 + BP
print(f"1. put {P1:.4f}: delta {dP:.4f}, bond {BP:.2f};  parity C - P = {delta * S0 + B - P1:.4f}, S - K/R = {S0 - K / R:.4f}")

# 2.
print("2. R = 1.25 > u: short the stock (receive 100), invest in the bond -> 125 >= 120 or 90 owed: riskless profit 5 or 35")

# 4.
for K4 in [90, 100, 110]:
    for r4 in [0.01, 0.05]:
        ce = binomial_price(S, K4, T, r4, sigma, 300)
        ca, _ = binomial_price(S, K4, T, r4, sigma, 300, "call", american=True)
        print(f"4. K {K4}, r {r4}: European {ce:.4f}, American {ca:.4f}")

In [ ]:
# 4. with a dividend: proportional 3% drop in the stock at the step nearest t = 0.5
def binomial_dividend(S, K, T, r, sigma, n, div, t_div, american):
    dt = T / n
    u_, d_ = np.exp(sigma * np.sqrt(dt)), np.exp(-sigma * np.sqrt(dt))
    R_ = np.exp(r * dt)
    q_ = (R_ - d_) / (u_ - d_)
    step_div = int(round(t_div / dt))
    factor = lambda step: (1 - div) if step >= step_div else 1.0
    k = np.arange(n + 1)
    V = np.maximum(S * factor(n) * u_**k * d_ ** (n - k) - K, 0)
    for step in range(n - 1, -1, -1):
        k = np.arange(step + 1)
        V = (q_ * V[1:] + (1 - q_) * V[:-1]) / R_
        if american:
            V = np.maximum(V, S * factor(step) * u_**k * d_ ** (step - k) - K)
    return V[0]

for K4 in [90, 100]:
    e_ = binomial_dividend(S, K4, T, 0.01, sigma, 300, 0.03, 0.5, False)
    a_ = binomial_dividend(S, K4, T, 0.01, sigma, 300, 0.03, 0.5, True)
    print(f"4. with 3% dividend, K {K4}: European {e_:.4f}, American {a_:.4f}")

# 5.
for n5 in [10, 100, 1000]:
    gaps = [binomial_price(S, K5, T, r, sigma, n5) - binomial_price(S, K5, T, r, sigma, n5, "put") - (S - K5 * np.exp(-r * T))
            for K5 in [80, 100, 120]]
    print(f"5. n {n5}: max |parity gap| {np.max(np.abs(gaps)):.1e}")

In [ ]:
# 6.
for n6 in [10, 100, 1000]:
    dt6 = T / n6
    u6 = np.exp(sigma * np.sqrt(dt6))
    q6 = (np.exp(r * dt6) - 1 / u6) / (u6 - 1 / u6)
    m6 = (2 * q6 - 1) * sigma * np.sqrt(dt6)
    v6 = 4 * q6 * (1 - q6) * sigma**2 * dt6
    print(f"6. n {n6}: mean/dt {m6 / dt6:.5f} (r - sigma^2/2 = {r - sigma**2 / 2:.5f}), var/dt {v6 / dt6:.5f} (sigma^2 = {sigma**2:.5f})")

# 7.
def digital_tree(n7):
    dt7 = T / n7
    u7 = np.exp(sigma * np.sqrt(dt7))
    R7 = np.exp(r * dt7)
    q7 = (R7 - 1 / u7) / (u7 - 1 / u7)
    k7 = np.arange(n7 + 1)
    return R7 ** (-n7) * np.sum(stats.binom.pmf(k7, n7, q7) * (S * u7 ** (2 * k7 - n7) > K))
d2 = (np.log(S / K) + (r - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
dig_bs = np.exp(-r * T) * stats.norm.cdf(d2)
dig = np.array([digital_tree(n7) for n7 in range(1, 201)])
print(f"7. Black-Scholes digital {dig_bs:.4f}; tree n = 199: {dig[198]:.4f}, n = 200: {dig[199]:.4f}")

**1.** $P_u = 0$, $P_d = 10$: $\Delta = -10/30 = -0.333$ shares (short), $B = (1.2\cdot 10)/(0.3\cdot 1.05) = 38.10$; $P = -33.33 + 38.10 = 4.762$. Parity:
$9.524 - 4.762 = 4.762 = 100 - 100/1.05$.

**2.** With $R = 1.25 > u$, the bond beats the stock in every state: short one share (receive 100), invest 100 in the bond (worth 125), and buy the
share back for 120 or 90: profit 5 or 35 with no risk. No-arbitrage requires $d < R < u$.

**3.** The option can be replicated with the stock and the bond whatever the probability of an up-move. The analysts' 80% view, if shared by the
market, is already in today's stock price; if it is a private view, the right trade is in the stock (or a leveraged position in it), but the option
is still worth exactly its replication cost relative to that stock price.

**4.** Without dividends the American and European call prices are identical for all strikes and rates. With a 3% dividend in mid-year, the
American call is worth more: exercising just before the dividend can be optimal for deep in-the-money calls.

**5.** The parity gap is zero up to rounding for all $n$ and strikes: parity is a model-free relation that holds exactly in the tree.

**6.** $E_q[\ln(S_{t+1}/S_t)] = (2q - 1)\sigma\sqrt{\Delta t}$ and $\operatorname{Var}_q = 4q(1-q)\sigma^2\Delta t$. Expanding $q = \frac12 + \frac{r - \sigma^2/2}{2\sigma}\sqrt{\Delta t} + O(\Delta t)$ gives a mean of $(r - \sigma^2/2)\Delta t$ and a variance of
$\sigma^2\Delta t$ up to $O(\Delta t^{3/2})$ and $O(\Delta t^2)$; the printed ratios converge to these values.

**7.** The digital payoff jumps from 0 to 1 at the strike, so the price depends on which side of $K$ the nearest tree node falls; as $n$ changes the
node grid moves across the strike, and the price oscillates by an amount that shrinks only slowly. The ordinary call's payoff is continuous at the
strike, so the error is smaller.

## 9. Conclusion

- In a one-period model an option is replicated by $\Delta = (C_u - C_d)/((u-d)S)$ shares and a bond; no arbitrage fixes its price.
- The price is the discounted expectation under $q = (R - d)/(u - d)$; under $q$ the stock earns the risk-free rate, and the real probability does not appear.
- In $n$ steps, backward induction or the binomial formula gives European prices; CRR trees converge to Black–Scholes as $n$ grows.
- American options compare exercise and continuation in each node; puts can be exercised early, calls on non-dividend stocks never.
- Put–call parity holds exactly in the tree.

Next: [10/02 From random walk to Brownian motion](./02_from_random_walk_to_brownian_motion.ipynb).